# Дообучение YOLO11-OBB на MAR20 (военные самолёты, 20 типов)

**Настройки ноутбука Kaggle:** Accelerator — GPU T4 x2 или P100, Internet — On (нужен подтверждённый телефон).

**Запуск в фоне:** Save Version → Save & Run All. Результаты (веса, метрики, графики) окажутся в Output версии.

Старт с весов `yolo11s-obb.pt` (Ultralytics, DOTA-v1.0, AGPL-3.0) — трансферное обучение.
Датасет MAR20 — CC BY-NC 4.0, https://gcheng-nwpu.github.io/. Если зеркало на Hugging Face недоступно или неполное,
скачайте официальный архив, загрузите как Kaggle Dataset и укажите путь в `MAR20_SRC`.

In [ ]:
REPO_URL = "https://github.com/<user>/<repo>.git"  # ← заменить на репозиторий команды
MAR20_SRC = None          # путь к распакованному MAR20 в /kaggle/input/...; None — скачать зеркало с Hugging Face
MODEL = "yolo11s-obb.pt"  # yolo11m-obb.pt — точнее, но ~2x медленнее
EPOCHS, IMGSZ, BATCH = 60, 800, 16  # снимки MAR20 800x800
RUN_NAME = "mar20_s_800"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q ultralytics==8.4.152 huggingface_hub

In [ ]:
if MAR20_SRC is None:
    from huggingface_hub import snapshot_download
    MAR20_SRC = snapshot_download(
        "Alex5666/Military-Aircraft-Recognition-dataset", repo_type="dataset",
        local_dir="/kaggle/working/MAR20_src",
    )
!python scripts/prepare_mar20.py --src "$MAR20_SRC" --out /kaggle/working/mar20_yolo

In [ ]:
import torch
DEVICE = ",".join(str(i) for i in range(torch.cuda.device_count())) or "cpu"
print("device:", DEVICE)
!python scripts/train.py --data /kaggle/working/mar20_yolo/mar20.yaml --weights $MODEL --name $RUN_NAME \
    --epochs $EPOCHS --imgsz $IMGSZ --batch $BATCH --device $DEVICE --workers 4

In [ ]:
import glob, shutil
from pathlib import Path
from IPython.display import Image, Markdown, display

export = Path("/kaggle/working/export")
export.mkdir(exist_ok=True)
run_dir = Path("runs/obb") / RUN_NAME
for f in [f"weights/{RUN_NAME}.pt", *glob.glob(f"outputs/eval/{RUN_NAME}_*.*"), run_dir / "results.csv", *run_dir.glob("*.png")]:
    shutil.copy2(f, export)
for md in glob.glob(f"outputs/eval/{RUN_NAME}_*.md"):
    display(Markdown(Path(md).read_text()))
display(Image(str(run_dir / "results.png")))
print(sorted(p.name for p in export.iterdir()))